In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from pyvis.network import Network
import json
import os

# Create directories for outputs if they don't exist
os.makedirs('results', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('visualizations', exist_ok=True)

# Set plotting style
sns.set_theme(style="whitegrid")

In [4]:
edge_df = pd.read_csv('../data/processed/graph_edges.csv')

# Task 2: Build a directed graph using NetworkX
G = nx.from_pandas_edgelist(
    edge_df, 
    source='source', 
    target='target', 
    create_using=nx.DiGraph()
)

print(f"Directed Graph created successfully.")
print(G)

Directed Graph created successfully.
DiGraph with 35777 nodes and 137821 edges


In [5]:
# Task 3: Compute basic graph metrics
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()

# Average degree (For directed graphs: average in-degree = average out-degree = edges/nodes)
avg_degree = num_edges / num_nodes if num_nodes > 0 else 0

# Graph density
density = nx.density(G)

# Clustering coefficient
clustering_coeff = nx.average_clustering(G)

# Connected components
num_wcc = nx.number_weakly_connected_components(G)
num_scc = nx.number_strongly_connected_components(G)

# Compile statistics into a dictionary
graph_stats = {
    "Number of Nodes": num_nodes,
    "Number of Edges": num_edges,
    "Average Degree": avg_degree,
    "Graph Density": density,
    "Average Clustering Coefficient": clustering_coeff,
    "Weakly Connected Components": num_wcc,
    "Strongly Connected Components": num_scc
}

# Display statistics
stats_df = pd.DataFrame(list(graph_stats.items()), columns=['Metric', 'Value'])
display(stats_df)

,Metric,Value
0,Number of Nodes,35777.000000
1,Number of Edges,137821.000000
2,Average Degree,3.852223
3,Graph Density,0.000108
4,Average Clustering Coefficient,0.139302
5,Weakly Connected Components,497.000000
6,Strongly Connected Components,24072.000000


In [ ]:
# Task 4: Compute Centralities
print("Computing Centrality Measures...")

# Degree Centrality
in_degree_cent = nx.in_degree_centrality(G)
out_degree_cent = nx.out_degree_centrality(G)
degree_cent = nx.degree_centrality(G)

# Betweenness and Closeness Centrality
betweenness_cent = nx.betweenness_centrality(G)
closeness_cent = nx.closeness_centrality(G)

# Aggregate results into a DataFrame
centrality_df = pd.DataFrame({
    'Subreddit Node': list(G.nodes()),
    'In-Degree Centrality': [in_degree_cent[n] for n in G.nodes()],
    'Out-Degree Centrality': [out_degree_cent[n] for n in G.nodes()],
    'Total Degree Centrality': [degree_cent[n] for n in G.nodes()],
    'Betweenness Centrality': [betweenness_cent[n] for n in G.nodes()],
    'Closeness Centrality': [closeness_cent[n] for n in G.nodes()]
})

# Sort by most influential (Betweenness)
centrality_df = centrality_df.sort_values(by='Betweenness Centrality', ascending=False)
display(centrality_df.head())

Computing Centrality Measures...


In [ ]:
# Task 5: Plotting

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Degree Distribution
in_degrees = [d for n, d in G.in_degree()]
sns.histplot(in_degrees, bins=15, ax=axes[0], color='skyblue', kde=True)
axes[0].set_title('In-Degree Distribution')
axes[0].set_xlabel('In-Degree (Number of Incoming Edges)')
axes[0].set_ylabel('Frequency')

# 2. Centrality Comparisons (Degree vs Betweenness)
sns.scatterplot(
    data=centrality_df, 
    x='Total Degree Centrality', 
    y='Betweenness Centrality', 
    alpha=0.7, 
    ax=axes[1],
    color='coral'
)
axes[1].set_title('Degree vs Betweenness Centrality')

# 3. Centrality Comparisons (Closeness vs Betweenness)
sns.scatterplot(
    data=centrality_df, 
    x='Closeness Centrality', 
    y='Betweenness Centrality', 
    alpha=0.7, 
    ax=axes[2],
    color='seagreen'
)
axes[2].set_title('Closeness vs Betweenness Centrality')

plt.tight_layout()
plt.savefig('visualizations/centrality_analysis.png')
plt.show()

In [ ]:
# Get basic communities (Undirected required for basic greedy modularity)
G_undirected = G.to_undirected()
communities = list(nx.community.greedy_modularity_communities(G_undirected))

# Get top 3 largest communities
top_communities = sorted(communities, key=len, reverse=True)[:3]

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, k=0.5, seed=42)

# Draw background nodes and edges
nx.draw_networkx_nodes(G, pos, node_size=50, node_color='lightgray', alpha=0.5)
nx.draw_networkx_edges(G, pos, alpha=0.2, edge_color='gray', arrows=True)

# Highlight top communities
colors = ['#FF595E', '#1982C4', '#8AC926']
for i, comm in enumerate(top_communities):
    nx.draw_networkx_nodes(
        G, pos, 
        nodelist=list(comm), 
        node_color=colors[i], 
        node_size=150, 
        label=f'Community {i+1}'
    )

# Add labels only for nodes in the top communities to avoid clutter
labels = {node: node for comm in top_communities for node in comm}
nx.draw_networkx_labels(G, pos, labels, font_size=8, font_color='black')

plt.title("Subreddit Network: Top 3 Influential Communities")
plt.legend()
plt.axis('off')
plt.savefig('visualizations/top_communities_static.png')
plt.show()

In [ ]:
# Task 6: Create Interactive Graph Visualization using PyVis
# Render top 100 most influential nodes (by betweenness) to keep browser smooth
top_nodes = centrality_df.head(100)['Subreddit Node'].tolist()
G_sub = G.subgraph(top_nodes)

# Initialize PyVis network
net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", directed=True)

# Compute PyVis physics layout
net.from_nx(G_sub)
net.barnes_hut(gravity=-8000)

# Save interactive graph to HTML
html_path = "visualizations/interactive_network.html"
net.write_html(html_path)
print(f"Interactive PyVis network saved to: {html_path}")

In [ ]:
# Task 7: Save Results as CSVs
centrality_df.to_csv('results/centrality_results.csv', index=False)
stats_df.to_csv('results/graph_statistics.csv', index=False)
print("Saved 'centrality_results.csv' and 'graph_statistics.csv' to results/ directory.")

# Task 8: Export graph object for other members
# Using GraphML so Community Detection and Toxicity Leads can easily load it later
nx.write_graphml(G, 'data/processed/directed_graph.graphml')
print("Graph object exported to 'data/processed/directed_graph.graphml'.")
print("Pipeline complete. Ready for downstream leads!")